# ***AI Agents***

## ***Installing and Importing Libraries***

In [1]:
!pip install pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## ***Loading Dataset***

In [3]:
dataset_url = "https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/resolve/main/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"

df = pd.read_csv(dataset_url)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


## ***Understanding Dataset***

In [8]:
print("Dataset Columns :")
print(df.columns.tolist())

print("\nDataset information :")
df.info()

print("\nAvailable Categories :")
print(df['category'].unique())

print("\nAvailable Intents :")
print(df['intent'].unique())

Dataset Columns :
['flags', 'instruction', 'category', 'intent', 'response']

Dataset information :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB

Available Categories :
['ORDER' 'SHIPPING' 'CANCEL' 'INVOICE' 'PAYMENT' 'REFUND' 'FEEDBACK'
 'CONTACT' 'ACCOUNT' 'DELIVERY' 'SUBSCRIPTION']

Available Intents :
['cancel_order' 'change_order' 'change_shipping_address'
 'check_cancellation_fee' 'check_invoice' 'check_payment_methods'
 'check_refund_policy' 'complaint' 'contact_customer_service'
 'contact_human_agent' 'create_account' 'delete_account'
 'delivery_options' 'delivery_period' 'edit_account' 'get_inv

## ***Preparing Data***

In [9]:
df = df.dropna(subset=['instruction', 'response'])

df['instruction'] = df['instruction'].astype(str)
df['response'] = df['response'].astype(str)

df = df.reset_index(drop=True)

print("Cleaned dataset :", df.shape)

Cleaned dataset : (26872, 5)


## ***Vectorizing Data***

In [12]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2)
)

question_vectors = vectorizer.fit_transform(
    df['instruction']
)

print("Knowledge base created!")
print("Vector shape :", question_vectors.shape)

Knowledge base created!
Vector shape : (26872, 12979)


## ***Craete FQA Search tool***

In [13]:
def search_faq(query):

  query_vector = vectorizer.transform([query])

  similarities = cosine_similarity(
      query_vector,
      question_vectors
  )[0]

  best_index = similarities.argmax()

  result = df.iloc[best_index]

  return {
      "question" : result['instruction'],
      "answer" : result['response'],
      "category" : result['category'],
      "intent" : result['intent'],
      "score" : similarities[best_index]
   }

## ***Test the search tool***

In [15]:
result = search_faq("How can I get my money back?")

print("Matched question :",result['question'])
print("category :",result['category'])
print("intent :",result['intent'])
print("Similarity :",result['score'])
print("\nAnswer:",result['answer'])

Matched question : get money back
category : REFUND
intent : get_refund
Similarity : 1.0

Answer: I've taken note that you're seeking assistance in getting your money back, and I'm here to provide you with the necessary steps to ensure a successful refund process. To initiate the refund, you can start by reaching out to our customer support team at {{Customer Support Phone Number}} or through the Live Chat on our website at {{Website URL}}. They will guide you through the process and help you with any additional steps required, such as providing the necessary documentation or filling out a refund request form. Rest assured that we value your satisfaction and will endeavor to resolve this matter in a timely and efficient manner.


## ***Create calculator tool***

In [19]:
import ast
import operator

operations = {
    ast.Add : operator.add,
    ast.Sub : operator.sub,
    ast.Mult : operator.mul,   # Fixed: 'Multi' -> 'Mult'
    ast.Div : operator.truediv
}

def calculate(expression):
    def evaluate(node):
        if isinstance(node, ast.Expression):
            return evaluate(node.body)

        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value

        if isinstance(node, ast.BinOp) and type(node.op) in operations:
            return operations[type(node.op)](
                evaluate(node.left),
                evaluate(node.right)
            )

        raise ValueError("Unsupported calculation!")


    tree = ast.parse(expression, mode='eval')

    return evaluate(tree)

## ***Craete a support escalation tool***

In [20]:
def contact_support():

  return(
      "This query needs human assistance. "
      "Please contact the customer support time."
  )

## ***Craete the agentic tool collection***

In [21]:
tools = {
    "search_faq": search_faq,
    "calculate": calculate,
    "contact_support": contact_support
}

print("Available agent tools :")

for tool_name in tools:
  print("-", tool_name)

Available agent tools :
- search_faq
- calculate
- contact_support


## ***Craete the decision-making function***

In [27]:
def decide_tool(query):

  q = query.lower()

  # Check for arithematic queries
  if re.search(r'\d+\s*[\+\-\*/]\s*\d+', q):
    return "calculate"

  # Check for human assistance
  if any(word in q for word in [
      "human", "agent", "representative",
      "speak to someone", "talk to someone"
  ]):
      return "contact_support"

  # default action
  return "search_faq"

## ***Extract calculation expressions***

In [28]:
def extract_expression(query):

  match = re.search(
      r'\d+(?:\.\d+)?(?:\s*[\+\-\*/]\s*\d+(?:\.\d+)?)+',
      query
  )

  if match:
    return match.group()

  return None

## ***Build AI Agent***

In [30]:
conversation_memory = []

def ai_agent(user_query):
    print("\nUSER :", user_query)

    # Step 1. Understand and decide
    selected_tool = decide_tool(user_query)
    print("AGENT DECISION :", selected_tool)

    # Step 2. Execute selected tool
    if selected_tool == "calculate":
        expression = extract_expression(user_query)
        if expression is None:
            response = "Please provide a valid calculation."
        else:
            result = tools["calculate"](expression)
            response = f"The result is {result}"

    elif selected_tool == "contact_support":
        response = tools['contact_support']()

    else:
        result = tools['search_faq'](user_query)
        print("MATCHED INTENT :", result['intent'])
        print("SIMILARITY :", round(result['score'], 3))

        if result['score'] < 0.10:
            response = (
                "I could not find a sufficiently relevant "
                "answer, please contact customer support."
            )
        else:
            response = result['answer']

    # Step 3. Store conversation
    conversation_memory.append({
        "user": user_query,
        "agent": response,
        "tool": selected_tool
    })

    # Step 4. Return final response
    print("\nAGENT RESPONSE :", response)
    return response

## ***Test the Agent***

In [33]:
ai_agent("Now can I track my refund?")


USER : Now can I track my refund?
AGENT DECISION : search_faq
MATCHED INTENT : track_refund
SIMILARITY : 1.0

AGENT RESPONSE : I'm on the same page, the importance of keeping track of your refund. Let me look into it for you and provide you with a detailed update on its status. While I review your case, may I ask if you have any specific concerns or questions regarding your refund? Your input is valuable, and I want to ensure that all your queries are addressed. Rest assured, I'll do my utmost to expedite the process and keep you informed every step of the way.


"I'm on the same page, the importance of keeping track of your refund. Let me look into it for you and provide you with a detailed update on its status. While I review your case, may I ask if you have any specific concerns or questions regarding your refund? Your input is valuable, and I want to ensure that all your queries are addressed. Rest assured, I'll do my utmost to expedite the process and keep you informed every step of the way."

In [32]:
ai_agent("I want to cancel my order")


USER : I want to cancel my order
AGENT DECISION : search_faq
MATCHED INTENT : cancel_order
SIMILARITY : 0.89

AGENT RESPONSE : I've understood, you would like to cancel your order associated with the purchase number {{Order Number}}. We apologize for any inconvenience this may have caused. To assist you with the cancellation process, please follow the steps below:

1. Log in to your {{Online Company Portal Info}} using your credentials.
2. Navigate to the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.
3. Locate the order with the order number {{Order Number}} and click on it for more details.
4. Look for the '{{Online Order Interaction}}' option associated with the purchase and select it.
5. Follow any additional instructions or prompts provided by the system to confirm the cancellation.

For any further assistance or if you have any questions, please feel free to reach out to our customer support team. They are available during {{Customer Support Hours}} at

"I've understood, you would like to cancel your order associated with the purchase number {{Order Number}}. We apologize for any inconvenience this may have caused. To assist you with the cancellation process, please follow the steps below:\n\n1. Log in to your {{Online Company Portal Info}} using your credentials.\n2. Navigate to the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.\n3. Locate the order with the order number {{Order Number}} and click on it for more details.\n4. Look for the '{{Online Order Interaction}}' option associated with the purchase and select it.\n5. Follow any additional instructions or prompts provided by the system to confirm the cancellation.\n\nFor any further assistance or if you have any questions, please feel free to reach out to our customer support team. They are available during {{Customer Support Hours}} at {{Customer Support Phone Number}} or through the Live Chat feature on our {{Website URL}}. We appreciate your understa

In [34]:
ai_agent("what is 250 + 240")


USER : what is 250 + 240
AGENT DECISION : calculate

AGENT RESPONSE : The result is 490


'The result is 490'

In [37]:
ai_agent("what is 20 / 5")


USER : what is 20 / 5
AGENT DECISION : calculate

AGENT RESPONSE : The result is 4.0


'The result is 4.0'

In [38]:
ai_agent("I want to speak to a human agent")


USER : I want to speak to a human agent
AGENT DECISION : contact_support

AGENT RESPONSE : This query needs human assistance. Please contact the customer support time.


'This query needs human assistance. Please contact the customer support time.'

## ***Demonstrate Agent Memory***

In [39]:
print("Conversation History:\n")

for item in conversation_memory:

  print("user:", item['user'])
  print("Tools used:", item['tool'])
  print("Agent", item['agent'])
  print("-" * 50)

Conversation History:

user: Now can i track my refund?
Tools used: search_faq
Agent I'm on the same page, the importance of keeping track of your refund. Let me look into it for you and provide you with a detailed update on its status. While I review your case, may I ask if you have any specific concerns or questions regarding your refund? Your input is valuable, and I want to ensure that all your queries are addressed. Rest assured, I'll do my utmost to expedite the process and keep you informed every step of the way.
--------------------------------------------------
user: I want to cancel my order
Tools used: search_faq
Agent I've understood, you would like to cancel your order associated with the purchase number {{Order Number}}. We apologize for any inconvenience this may have caused. To assist you with the cancellation process, please follow the steps below:

1. Log in to your {{Online Company Portal Info}} using your credentials.
2. Navigate to the '{{Online Order Interaction}}

## ***Create continuous chat interface***

In [40]:
print("COUSTOMER SUPPORT AI AGENT")
print("Type 'exit' to stop.\n")

while True:

  user_input = input("You: ")

  if user_input.lower() == "exit":
    print("Agent: Thank you for contacting us!")
    break

  ai_agent(user_input)

COUSTOMER SUPPORT AI AGENT
Type 'exit' to stop.

You: how order football

USER : how order football
AGENT DECISION : search_faq
MATCHED INTENT : place_order
SIMILARITY : 1.0

AGENT RESPONSE : I'm delighted to assist you in finding the right place to order our products! To place an order, you can visit our website at {{Website URL}}. Our website is designed to provide you with a user-friendly and seamless ordering experience. Simply search for the items you wish to purchase, add them to your cart, and proceed to the checkout page to complete your order. Our website also offers secure payment options to ensure a smooth and protected transaction. If you have any questions or need further assistance during the ordering process, our customer support team is always available to help you. Happy shopping!
You: stop

USER : stop
AGENT DECISION : search_faq
MATCHED INTENT : cancel_order
SIMILARITY : 0.0

AGENT RESPONSE : I could not find a sufficiently relevant answer, please contact customer su

## ***GRADIO***

In [41]:
!pip install gradio

In [43]:
import gradio as gr

def gradio_chat(message, history):
    response = ai_agent(message)
    return response

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="Customer Support AI Agent",
    description="Ask questions and interact with the AI agent.",
    chatbot=gr.Chatbot(height=450),
    textbox=gr.Textbox(
        placeholder="Ask your question here...",
        container=False,
        scale=7
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://652733ddcaba9a894d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
